# 08 — Qualitative Comparison (Pulau Ubin)

Place model predictions for Pulau Ubin side-by-side with the National Parks Board
mangrove mask and the original GMW labels.

**Pipeline step:** Qualitative comparison (step 7 from the paper workflow).

> _"A qualitative comparison stage places the model's predictions for Pulau Ubin
> side by side with the National Parks Board mangrove mask. This comparison is
> strictly qualitative: the NParks map dates from several years ago and reflects
> a past state of the ecosystem, so it cannot serve as ground truth."_

**Inputs:**
- Pulau Ubin AlphaEarth embedding (`.npz`)
- Trained model checkpoint (`.ckpt`)
- NParks mangrove mask (image)
- GMW extent for Pulau Ubin

**Outputs:**
- Side-by-side visual comparison

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

from model.training.models import ViTClassifier
from model.training.modules import LitModule
from model.training.gradcam import GradCAM

## 1. Configuration

Adjust paths to match your local setup.

In [ ]:
# ---- USER CONFIG ----
CHECKPOINT_PATH     = '../../output/checkpoints/best_model.ckpt'
PULAU_UBIN_NPZ      = '../../output/dataset/pulau_ubin_embedding.npz'  # AlphaEarth embedding for Pulau Ubin
NPARKS_MASK_PATH    = None  # Path to NParks mangrove mask image (PNG/JPG)
GMW_VISUAL_PATH     = None  # Path to GMW visual for comparison
SENTINEL_RGB_PATH   = None  # Path to Sentinel-2 RGB image of Pulau Ubin

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CATEGORY_NAMES = ['1-20%', '21-40%', '41-60%', '61-80%', '81-100%']

## 2. Load Model & Predict on Pulau Ubin

In [ ]:
net = ViTClassifier(in_channels=64, out_features=len(CATEGORY_NAMES))

if os.path.exists(CHECKPOINT_PATH):
    lit = LitModule.load_from_checkpoint(CHECKPOINT_PATH, net=net)
    model = lit.net
    print(f'Loaded checkpoint from {CHECKPOINT_PATH}')
else:
    model = net
    print('WARNING: no checkpoint found — using untrained model.')

model = model.to(DEVICE).eval()

In [ ]:
# Load Pulau Ubin embedding
if os.path.exists(PULAU_UBIN_NPZ):
    data = np.load(PULAU_UBIN_NPZ)
    key = 'embeddings' if 'embeddings' in data else list(data.keys())[0]
    emb_np = data[key].astype(np.float32)
    emb_tensor = torch.from_numpy(emb_np).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(emb_tensor)
        probs = torch.softmax(logits, dim=1)[0]
        pred_class = probs.argmax().item()
        confidence = probs[pred_class].item()

    print(f'Prediction: {CATEGORY_NAMES[pred_class]} (confidence: {confidence:.1%})')
    print(f'All probabilities: {dict(zip(CATEGORY_NAMES, [f"{p:.1%}" for p in probs.cpu().numpy()]))}')
else:
    print(f'Embedding file not found: {PULAU_UBIN_NPZ}')
    print('Download the Pulau Ubin embedding using notebook 03_download_embeddings.ipynb.')

## 3. Side-by-Side Comparison

Visual comparison of: Sentinel-2 RGB | GMW Classification | Model Prediction + Grad-CAM | NParks Mask

In [ ]:
def embeddings_to_pseudo_rgb(emb):
    rgb = emb[:3].transpose(1, 2, 0).copy()
    for c in range(3):
        lo, hi = np.percentile(rgb[:, :, c], [2, 98])
        rgb[:, :, c] = np.clip((rgb[:, :, c] - lo) / (hi - lo + 1e-8), 0, 1)
    return rgb


panels = []
titles = []

# Panel 1: Sentinel-2 or pseudo-RGB
if SENTINEL_RGB_PATH and os.path.exists(SENTINEL_RGB_PATH):
    panels.append(plt.imread(SENTINEL_RGB_PATH))
    titles.append('Sentinel-2 RGB')
elif os.path.exists(PULAU_UBIN_NPZ):
    panels.append(embeddings_to_pseudo_rgb(emb_np))
    titles.append('Pseudo-RGB (bands 0-2)')

# Panel 2: GMW visual
if GMW_VISUAL_PATH and os.path.exists(GMW_VISUAL_PATH):
    panels.append(plt.imread(GMW_VISUAL_PATH))
    titles.append('GMW Classification')

# Panel 3: Grad-CAM overlay
if os.path.exists(PULAU_UBIN_NPZ):
    gradcam = GradCAM(model, model.blocks[-1].norm1)
    heatmap = gradcam(emb_tensor, target_class=pred_class,
                      img_size=(emb_np.shape[1], emb_np.shape[2]))
    rgb = embeddings_to_pseudo_rgb(emb_np)
    cmap = plt.cm.jet
    overlay = 0.5 * rgb + 0.5 * cmap(heatmap)[:, :, :3]
    panels.append(np.clip(overlay, 0, 1))
    titles.append(f'Pred: {CATEGORY_NAMES[pred_class]} + Grad-CAM')
    gradcam.remove_hooks()

# Panel 4: NParks mask
if NPARKS_MASK_PATH and os.path.exists(NPARKS_MASK_PATH):
    panels.append(plt.imread(NPARKS_MASK_PATH))
    titles.append('NParks Mangrove Mask')

# Plot
if panels:
    n = len(panels)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 6))
    if n == 1:
        axes = [axes]
    for ax, img, title in zip(axes, panels, titles):
        ax.imshow(img)
        ax.set_title(title, fontsize=12)
        ax.axis('off')
    plt.suptitle('Pulau Ubin — Qualitative Comparison', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No visual data available. Please set the paths in section 1.')

## 4. Discussion

Points to assess qualitatively:

1. **Boundary delineation**: Does the model improve mangrove boundaries compared to GMW,
   especially in tidal transition zones?
2. **Fringing mangroves**: Are narrow mangrove strips along the coast better captured?
3. **False positives**: Does the model confuse dense terrestrial vegetation with mangroves?
4. **Tidal zones**: Are temporarily submerged areas correctly classified?

Note: The NParks map dates from several years ago and cannot serve as ground truth—
this comparison is **strictly qualitative**.